# Evaluation on LLaMA 3.1-8B + Instruction-Tuned using CounterBench for reasoning tasks

In [1]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 14.8 MB/s eta 0:00:00


In [2]:
import json
import torch
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import classification_report

In [3]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
model_name = "meta-llama/Llama-3.1-8B-Instruct"

In [5]:
bitsandbytes_config = BitsAndBytesConfig(load_in_4bit=True,
                                         bnb_4bit_compute_dtype=torch.float16,
                                         bnb_4bit_quant_type='nf4',
                                         bnb_4bit_use_double_quant=True)

In [7]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name,
                                             quantization_config=bitsandbytes_config,
                                             device_map="auto",
                                             offload_folder="offload")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [8]:
def get_prompt(instruction,questions,contexts):
  prompts=[]
  for i in range(len(questions)):
    prompt = f"{instruction}\nContext:{contexts[i]}\nQuestion:{questions[i]}\nAnswer:"
    prompts.append(prompt)
  return prompts

In [23]:
def training(prompts,num_tokens=1):
  predictions=[]
  for i in range(len(prompts)):
    input_tokens = tokenizer(prompts[i],return_tensors="pt")
    input_ids = input_tokens["input_ids"].to("cuda:0")
    attention_mask = input_tokens["attention_mask"].to("cuda:0")
    output_tokens = model.generate(input_ids=input_ids,attention_mask=attention_mask,max_new_tokens=num_tokens,pad_token_id=tokenizer.eos_token_id,
                                   eos_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
    pred_answer = response.split("Answer:")[1].split("Explanation:")[0]
    print(f"Iteration {i}\n")
    print(f"Response: {pred_answer}")
    pred_answer = pred_answer.replace("\n","").strip()
    predictions.append(pred_answer)

  return predictions

In [10]:
ds = load_dataset(
    "json",
    data_files=[
        "hf://datasets/CounterBench/CounterBench/data_balanced_alpha_V1.json",
        "hf://datasets/CounterBench/CounterBench/data_balanced_backdoor_V2.json"
    ]
)

print(ds)
print(ds["train"].features)

data_balanced_alpha_V1.json: 0.00B [00:00, ?B/s]

data_balanced_backdoor_V2.json: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'given_info', 'answer', 'type', 'question_id', 'meta'],
        num_rows: 1200
    })
})
{'question': Value('string'), 'given_info': Value('string'), 'answer': Value('string'), 'type': Value('string'), 'question_id': Value('string'), 'meta': {'graph_id': Value('string'), 'model_id': Value('int64'), 'query_type': Value('string'), 'rung': Value('int64'), 'story_id': Value('string')}}


In [11]:
dataset = ds["train"].to_pandas()

In [12]:
dataset

,question,given_info,answer,type,question_id,meta
0,Would Lumbo occur if not Ziklo instead of Ziklo?,"We know that Ziklo causes Blaf, Blaf causes Tr...",no,basic,0,"{'graph_id': 'graph5', 'model_id': 0, 'query_t..."
1,Would Zlim occur if not Glent instead of Glent?,"We know that Glent causes Razz, Razz causes Pe...",no,basic,200,"{'graph_id': 'graph5', 'model_id': 1, 'query_t..."
2,Would Klep occur if not Praf instead of Praf?,"We know that Praf causes Vank, Vank causes Scu...",no,basic,400,"{'graph_id': 'graph5', 'model_id': 2, 'query_t..."
3,Would Jext occur if not Fizo instead of Fizo?,"We know that Fizo causes Blorn, Blorn causes P...",no,basic,600,"{'graph_id': 'graph5', 'model_id': 3, 'query_t..."
4,Would Wrox occur if not Nuv instead of Nuv?,"We know that Nuv causes Splee, Splee causes Bl...",no,basic,800,"{'graph_id': 'graph5', 'model_id': 4, 'query_t..."
...,...,...,...,...,...,...
1195,Would Lumbo occur if not Ziklo instead of Ziklo?,"We know that Blaf causes Ziklo, Blaf causes Tr...",no,backdoor,39,"{'graph_id': 'graph9', 'model_id': 195, 'query..."
1196,Would Zlim occur if not Glent instead of Glent?,"We know that Razz causes Glent, Razz causes Pe...",no,backdoor,239,"{'graph_id': 'graph9', 'model_id': 196, 'query..."
1197,Would Klep occur if not Praf instead of Praf?,"We know that Vank causes Praf, Vank causes Scu...",no,backdoor,439,"{'graph_id': 'graph9', 'model_id': 197, 'query..."
1198,Would Jext occur if not Fizo instead of Fizo?,"We know that Blorn causes Fizo, Blorn causes P...",no,backdoor,639,"{'graph_id': 'graph9', 'model_id': 198, 'query..."


In [13]:
dataset['type'].value_counts()

,count
type,
basic,250
conditional,250
joint,250
nested,250
backdoor,200


In [14]:
intstruction = "Instruction: Answer the question strictly using the causal facts provided. Do not introduce any new information. For counterfactuals, simulate the scenario along the causal chain step by step. Only output the final answer as Yes or No. Do not write anything else."

## Basic Questions

In [15]:
dataset_basic = dataset[dataset['type']=="basic"]

In [16]:
dataset_basic

,question,given_info,answer,type,question_id,meta
0,Would Lumbo occur if not Ziklo instead of Ziklo?,"We know that Ziklo causes Blaf, Blaf causes Tr...",no,basic,0,"{'graph_id': 'graph5', 'model_id': 0, 'query_t..."
1,Would Zlim occur if not Glent instead of Glent?,"We know that Glent causes Razz, Razz causes Pe...",no,basic,200,"{'graph_id': 'graph5', 'model_id': 1, 'query_t..."
2,Would Klep occur if not Praf instead of Praf?,"We know that Praf causes Vank, Vank causes Scu...",no,basic,400,"{'graph_id': 'graph5', 'model_id': 2, 'query_t..."
3,Would Jext occur if not Fizo instead of Fizo?,"We know that Fizo causes Blorn, Blorn causes P...",no,basic,600,"{'graph_id': 'graph5', 'model_id': 3, 'query_t..."
4,Would Wrox occur if not Nuv instead of Nuv?,"We know that Nuv causes Splee, Splee causes Bl...",no,basic,800,"{'graph_id': 'graph5', 'model_id': 4, 'query_t..."
...,...,...,...,...,...,...
495,Would Lumbo occur if not Ziklo instead of Ziklo?,"We know that Ziklo causes Blaf and Trune, Trun...",no,basic,99,"{'graph_id': 'graph9', 'model_id': 495, 'query..."
496,Would Zlim occur if not Glent instead of Glent?,"We know that Glent causes Razz and Pex, Pex ca...",no,basic,299,"{'graph_id': 'graph9', 'model_id': 496, 'query..."
497,Would Klep occur if not Praf instead of Praf?,"We know that Praf causes Vank and Scud, Scud c...",no,basic,499,"{'graph_id': 'graph9', 'model_id': 497, 'query..."
498,Would Jext occur if not Fizo instead of Fizo?,"We know that Fizo causes Blorn and Plim, Plim ...",no,basic,699,"{'graph_id': 'graph9', 'model_id': 498, 'query..."


In [17]:
questions = dataset_basic["question"].values.tolist()

In [18]:
contexts = dataset_basic["given_info"].values.tolist()

In [19]:
true_answers = dataset_basic["answer"].values.tolist()

In [20]:
prompts = get_prompt(intstruction,questions,contexts)

In [21]:
prompts[0]

'Instruction: Answer the question strictly using the causal facts provided. Do not introduce any new information. For counterfactuals, simulate the scenario along the causal chain step by step. Only output the final answer as Yes or No. Do not write anything else.\nContext:We know that Ziklo causes Blaf, Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo.\nQuestion:Would Lumbo occur if not Ziklo instead of Ziklo?\nAnswer:'

In [24]:
predictions = training(prompts)

Iteration 0

Response: No
Iteration 1

Response: No
Iteration 2

Response:  No
Iteration 3

Response:  No
Iteration 4

Response: Yes
Iteration 5

Response: Yes
Iteration 6

Response:  No
Iteration 7

Response:  No
Iteration 8

Response: No
Iteration 9

Response: Yes
Iteration 10

Response: No
Iteration 11

Response: Yes
Iteration 12

Response:  No
Iteration 13

Response: No
Iteration 14

Response: No
Iteration 15

Response: No
Iteration 16

Response:  No
Iteration 17

Response: No
Iteration 18

Response:  No
Iteration 19

Response:  No
Iteration 20

Response:  No
Iteration 21

Response:  No
Iteration 22

Response: No
Iteration 23

Response:  No
Iteration 24

Response: Yes
Iteration 25

Response:  No
Iteration 26

Response:  No
Iteration 27

Response:  No
Iteration 28

Response: Yes
Iteration 29

Response: Yes
Iteration 30

Response: No
Iteration 31

Response: Yes
Iteration 32

Response:  No
Iteration 33

Response: No
Iteration 34

Response:  No
Iteration 35

Response:  No
Iteration 36


In [25]:
predictions

['No',
 'No',
 'No',
 'No',
 'Yes',
 'Yes',
 'No',
 'No',
 'No',
 'Yes',
 'No',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'No',
 'No',
 'No',
 'Yes',
 'Yes',
 'No',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'Yes',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'No',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'Yes',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'No',
 'No',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'No',
 'Yes',
 'No',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'No',
 'No',
 'Yes'

In [26]:
predictions_new = [elem.lower() for elem in predictions]
predictions_new

['no',
 'no',
 'no',
 'no',
 'yes',
 'yes',
 'no',
 'no',
 'no',
 'yes',
 'no',
 'yes',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'yes',
 'no',
 'no',
 'no',
 'yes',
 'yes',
 'no',
 'yes',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'yes',
 'yes',
 'no',
 'no',
 'no',
 'no',
 'yes',
 'yes',
 'yes',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'yes',
 'no',
 'yes',
 'no',
 'no',
 'no',
 'no',
 'no',
 'yes',
 'yes',
 'yes',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'yes',
 'no',
 'no',
 'yes',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'no',
 'yes',
 'no',
 'yes',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'yes',
 'no',
 'no',
 'yes'

In [27]:
print(classification_report(true_answers,predictions_new,zero_division=0))

              precision    recall  f1-score   support

          no       0.51      0.79      0.62       125
         yes       0.52      0.22      0.31       125

    accuracy                           0.51       250
   macro avg       0.51      0.51      0.46       250
weighted avg       0.51      0.51      0.46       250



## Conditional Questions

In [28]:
dataset_conditional = dataset[dataset['type']=="conditional"]

In [29]:
dataset_conditional

,question,given_info,answer,type,question_id,meta
25,Would Lumbo occur if not Ziklo instead of Ziklo?,"We know that Ziklo causes not Blaf, Blaf and T...",yes,conditional,5,"{'graph_id': 'graph5', 'model_id': 25, 'query_..."
26,Would Zlim occur if not Glent instead of Glent?,"We know that Glent causes not Razz, Razz and P...",yes,conditional,205,"{'graph_id': 'graph5', 'model_id': 26, 'query_..."
27,Would Klep occur if not Praf instead of Praf?,"We know that Praf causes not Vank, Vank and Sc...",yes,conditional,405,"{'graph_id': 'graph5', 'model_id': 27, 'query_..."
28,Would Jext occur if not Fizo instead of Fizo?,"We know that Fizo causes not Blorn, Blorn and ...",yes,conditional,605,"{'graph_id': 'graph5', 'model_id': 28, 'query_..."
29,Would Wrox occur if not Nuv instead of Nuv?,"We know that Nuv causes not Splee, Splee and B...",yes,conditional,805,"{'graph_id': 'graph5', 'model_id': 29, 'query_..."
...,...,...,...,...,...,...
435,Would Lumbo occur if not Ziklo instead of Ziklo?,"We know that Ziklo causes Blaf and Trune, Blaf...",no,conditional,87,"{'graph_id': 'graph7', 'model_id': 435, 'query..."
436,Would Zlim occur if not Glent instead of Glent?,"We know that Glent causes Razz and Pex, Razz a...",no,conditional,287,"{'graph_id': 'graph7', 'model_id': 436, 'query..."
437,Would Klep occur if not Praf instead of Praf?,"We know that Praf causes Vank and Scud, Vank a...",no,conditional,487,"{'graph_id': 'graph7', 'model_id': 437, 'query..."
438,Would Jext occur if not Fizo instead of Fizo?,"We know that Fizo causes Blorn and Plim, Blorn...",no,conditional,687,"{'graph_id': 'graph7', 'model_id': 438, 'query..."


In [30]:
questions = dataset_conditional["question"].values.tolist()

In [31]:
contexts = dataset_conditional["given_info"].values.tolist()

In [32]:
true_answers = dataset_conditional["answer"].values.tolist()

In [33]:
prompts = get_prompt(intstruction,questions,contexts)

In [34]:
prompts[0]

'Instruction: Answer the question strictly using the causal facts provided. Do not introduce any new information. For counterfactuals, simulate the scenario along the causal chain step by step. Only output the final answer as Yes or No. Do not write anything else.\nContext:We know that Ziklo causes not Blaf, Blaf and Trune together cause Vork, Vork causes Lumbo. We observed Trune\nQuestion:Would Lumbo occur if not Ziklo instead of Ziklo?\nAnswer:'

In [35]:
predictions = training(prompts)

Iteration 0

Response:  No
Iteration 1

Response: Yes
Iteration 2

Response:  Yes
Iteration 3

Response: Yes
Iteration 4

Response: Yes
Iteration 5

Response:  No
Iteration 6

Response: No
Iteration 7

Response:  No
Iteration 8

Response:  No
Iteration 9

Response:  Yes
Iteration 10

Response:  Yes
Iteration 11

Response:  No
Iteration 12

Response:  No
Iteration 13

Response: No
Iteration 14

Response:  No
Iteration 15

Response:  No
Iteration 16

Response: Yes
Iteration 17

Response:  No
Iteration 18

Response: No
Iteration 19

Response:  No
Iteration 20

Response:  No
Iteration 21

Response: Yes
Iteration 22

Response: No
Iteration 23

Response: No
Iteration 24

Response: Yes
Iteration 25

Response: Yes
Iteration 26

Response: No
Iteration 27

Response:  No
Iteration 28

Response: No
Iteration 29

Response:  No
Iteration 30

Response:  No
Iteration 31

Response:  Yes
Iteration 32

Response:  No
Iteration 33

Response:  No
Iteration 34

Response:  No
Iteration 35

Response: No
Iterat

In [36]:
predictions_new = [elem.lower() for elem in predictions]

In [37]:
print(classification_report(true_answers,predictions_new,zero_division=0))

              precision    recall  f1-score   support

          no       0.47      0.60      0.53       125
         yes       0.45      0.33      0.38       125

    accuracy                           0.46       250
   macro avg       0.46      0.46      0.45       250
weighted avg       0.46      0.46      0.45       250



## Joint Questions

In [38]:
dataset_joint = dataset[dataset['type']=="joint"]

In [39]:
dataset_joint

,question,given_info,answer,type,question_id,meta
500,Would Lumbo occur if not Ziklo and not Blaf?,"We know that Ziklo causes Blaf and Vork, Blaf ...",yes,joint,100,"{'graph_id': 'graph5', 'model_id': 500, 'query..."
501,Would Zlim occur if not Glent and not Razz?,"We know that Glent causes Razz and Zurn, Razz ...",yes,joint,300,"{'graph_id': 'graph5', 'model_id': 501, 'query..."
502,Would Klep occur if not Praf and not Vank?,"We know that Praf causes Vank and Wrenk, Vank ...",yes,joint,500,"{'graph_id': 'graph5', 'model_id': 502, 'query..."
503,Would Jext occur if not Fizo and not Blorn?,"We know that Fizo causes Blorn and Quaz, Blorn...",yes,joint,700,"{'graph_id': 'graph5', 'model_id': 503, 'query..."
504,Would Wrox occur if not Nuv and not Splee?,"We know that Nuv causes Splee and Druk, Splee ...",yes,joint,900,"{'graph_id': 'graph5', 'model_id': 504, 'query..."
...,...,...,...,...,...,...
745,Would Lumbo occur if not Ziklo and not Sline?,"We know that Ziklo causes Blaf, Blaf causes Tr...",yes,joint,149,"{'graph_id': 'graph7', 'model_id': 745, 'query..."
746,Would Zlim occur if not Glent and not Melf?,"We know that Glent causes Razz, Razz causes Pe...",yes,joint,349,"{'graph_id': 'graph7', 'model_id': 746, 'query..."
747,Would Klep occur if not Praf and not Yobb?,"We know that Praf causes Vank, Vank causes Scu...",yes,joint,549,"{'graph_id': 'graph7', 'model_id': 747, 'query..."
748,Would Jext occur if not Fizo and not Skul?,"We know that Fizo causes Blorn, Blorn causes P...",yes,joint,749,"{'graph_id': 'graph7', 'model_id': 748, 'query..."


In [40]:
questions = dataset_joint["question"].values.tolist()
contexts = dataset_joint["given_info"].values.tolist()
true_answers = dataset_joint["answer"].values.tolist()

In [41]:
prompts = get_prompt(intstruction,questions,contexts)

In [42]:
prompts[0]

'Instruction: Answer the question strictly using the causal facts provided. Do not introduce any new information. For counterfactuals, simulate the scenario along the causal chain step by step. Only output the final answer as Yes or No. Do not write anything else.\nContext:We know that Ziklo causes Blaf and Vork, Blaf causes not Trune, Trune or Vork causes Lumbo.\nQuestion:Would Lumbo occur if not Ziklo and not Blaf?\nAnswer:'

In [43]:
predictions = training(prompts)

Iteration 0

Response: No
Iteration 1

Response: Yes
Iteration 2

Response:  No
Iteration 3

Response: Yes
Iteration 4

Response:  Yes
Iteration 5

Response:  Yes
Iteration 6

Response: Yes
Iteration 7

Response:  Yes
Iteration 8

Response: Yes
Iteration 9

Response:  Yes
Iteration 10

Response:  No
Iteration 11

Response:  No
Iteration 12

Response:  No
Iteration 13

Response: Yes
Iteration 14

Response:  No
Iteration 15

Response: Yes
Iteration 16

Response: Yes
Iteration 17

Response: Yes
Iteration 18

Response:  Yes
Iteration 19

Response:  Yes
Iteration 20

Response: No
Iteration 21

Response: Yes
Iteration 22

Response:  No
Iteration 23

Response: Yes
Iteration 24

Response:  No
Iteration 25

Response:  No
Iteration 26

Response:  No
Iteration 27

Response:  No
Iteration 28

Response:  No
Iteration 29

Response: Yes
Iteration 30

Response: No
Iteration 31

Response:  No
Iteration 32

Response:  No
Iteration 33

Response:  No
Iteration 34

Response:  No
Iteration 35

Response: Yes

In [44]:
predictions_new = [elem.lower() for elem in predictions]

In [45]:
print(classification_report(true_answers,predictions_new,zero_division=0))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
          no       0.58      0.79      0.67       125
         yes       0.68      0.42      0.51       125

    accuracy                           0.60       250
   macro avg       0.42      0.40      0.39       250
weighted avg       0.63      0.60      0.59       250



## Nested Questions

In [46]:
dataset_nested = dataset[dataset['type']=="nested"]

In [47]:
dataset_nested

,question,given_info,answer,type,question_id,meta
750,"Assume not Ziklo, and based on this assumption...","We know that Ziklo causes Blaf, Trune and Vork...",yes,nested,150,"{'graph_id': 'graph9', 'model_id': 750, 'query..."
751,"Assume not Glent, and based on this assumption...","We know that Glent causes Razz, Pex and Zurn, ...",yes,nested,350,"{'graph_id': 'graph9', 'model_id': 751, 'query..."
752,"Assume not Praf, and based on this assumption,...","We know that Praf causes Vank, Scud and Wrenk,...",yes,nested,550,"{'graph_id': 'graph9', 'model_id': 752, 'query..."
753,"Assume not Fizo, and based on this assumption,...","We know that Fizo causes Blorn, Plim and Quaz,...",yes,nested,750,"{'graph_id': 'graph9', 'model_id': 753, 'query..."
754,"Assume not Nuv, and based on this assumption, ...","We know that Nuv causes Splee, Blen and Druk, ...",yes,nested,950,"{'graph_id': 'graph9', 'model_id': 754, 'query..."
...,...,...,...,...,...,...
995,"Assume not Ziklo, and based on this assumption...","We know that Ziklo causes Blaf and Trune, Trun...",yes,nested,199,"{'graph_id': 'graph8', 'model_id': 995, 'query..."
996,"Assume not Glent, and based on this assumption...","We know that Glent causes Razz and Pex, Pex ca...",yes,nested,399,"{'graph_id': 'graph8', 'model_id': 996, 'query..."
997,"Assume not Praf, and based on this assumption,...","We know that Praf causes Vank and Scud, Scud c...",yes,nested,599,"{'graph_id': 'graph8', 'model_id': 997, 'query..."
998,"Assume not Fizo, and based on this assumption,...","We know that Fizo causes Blorn and Plim, Plim ...",yes,nested,799,"{'graph_id': 'graph8', 'model_id': 998, 'query..."


In [48]:
questions = dataset_nested["question"].values.tolist()
contexts = dataset_nested["given_info"].values.tolist()
true_answers = dataset_nested["answer"].values.tolist()

In [49]:
prompts = get_prompt(intstruction,questions,contexts)

In [50]:
prompts[0]

'Instruction: Answer the question strictly using the causal facts provided. Do not introduce any new information. For counterfactuals, simulate the scenario along the causal chain step by step. Only output the final answer as Yes or No. Do not write anything else.\nContext:We know that Ziklo causes Blaf, Trune and Vork, not Vork causes Sline, Sline causes Frim, Frim or Vork causes Qado, Qado causes Jurf, and Jurf causes Lumbo.\nQuestion:Assume not Ziklo, and based on this assumption, further suppose not Sline. Would Lumbo occur?\nAnswer:'

In [51]:
predictions = training(prompts)

Iteration 0

Response:  No
Iteration 1

Response:  No
Iteration 2

Response:  No
Iteration 3

Response:  No
Iteration 4

Response:  No
Iteration 5

Response:  No
Iteration 6

Response: No
Iteration 7

Response:  No
Iteration 8

Response:  No
Iteration 9

Response:  No
Iteration 10

Response:  No
Iteration 11

Response:  No
Iteration 12

Response:  Yes
Iteration 13

Response:  No
Iteration 14

Response:  Yes
Iteration 15

Response:  Yes
Iteration 16

Response:  No
Iteration 17

Response: No
Iteration 18

Response:  No
Iteration 19

Response: Yes
Iteration 20

Response:  No
Iteration 21

Response:  No
Iteration 22

Response:  No
Iteration 23

Response:  No
Iteration 24

Response:  Yes
Iteration 25

Response:  No
Iteration 26

Response:  No
Iteration 27

Response:  No
Iteration 28

Response:  No
Iteration 29

Response: No
Iteration 30

Response:  Yes
Iteration 31

Response:  No
Iteration 32

Response:  No
Iteration 33

Response: Yes
Iteration 34

Response: No
Iteration 35

Response:  Yes


In [52]:
predictions_new = [elem.lower() for elem in predictions]

In [53]:
print(classification_report(true_answers,predictions_new,zero_division=0))

              precision    recall  f1-score   support

          no       0.51      0.70      0.59       125
         yes       0.53      0.34      0.41       125

    accuracy                           0.52       250
   macro avg       0.52      0.52      0.50       250
weighted avg       0.52      0.52      0.50       250



## Backdoor Questions

In [54]:
dataset_backdoor = dataset[dataset['type']=="backdoor"]

In [55]:
dataset_backdoor

,question,given_info,answer,type,question_id,meta
1000,Would Lumbo occur if not Ziklo instead of Ziklo?,"We know that Blaf causes Ziklo, Ziklo or Blaf ...",yes,backdoor,0,"{'graph_id': 'graph5', 'model_id': 0, 'query_t..."
1001,Would Zlim occur if not Glent instead of Glent?,"We know that Razz causes Glent, Glent or Razz ...",yes,backdoor,200,"{'graph_id': 'graph5', 'model_id': 1, 'query_t..."
1002,Would Klep occur if not Praf instead of Praf?,"We know that Vank causes Praf, Praf or Vank ca...",yes,backdoor,400,"{'graph_id': 'graph5', 'model_id': 2, 'query_t..."
1003,Would Jext occur if not Fizo instead of Fizo?,"We know that Blorn causes Fizo, Fizo or Blorn ...",yes,backdoor,600,"{'graph_id': 'graph5', 'model_id': 3, 'query_t..."
1004,Would Wrox occur if not Nuv instead of Nuv?,"We know that Splee causes Nuv, Nuv or Splee ca...",yes,backdoor,800,"{'graph_id': 'graph5', 'model_id': 4, 'query_t..."
...,...,...,...,...,...,...
1195,Would Lumbo occur if not Ziklo instead of Ziklo?,"We know that Blaf causes Ziklo, Blaf causes Tr...",no,backdoor,39,"{'graph_id': 'graph9', 'model_id': 195, 'query..."
1196,Would Zlim occur if not Glent instead of Glent?,"We know that Razz causes Glent, Razz causes Pe...",no,backdoor,239,"{'graph_id': 'graph9', 'model_id': 196, 'query..."
1197,Would Klep occur if not Praf instead of Praf?,"We know that Vank causes Praf, Vank causes Scu...",no,backdoor,439,"{'graph_id': 'graph9', 'model_id': 197, 'query..."
1198,Would Jext occur if not Fizo instead of Fizo?,"We know that Blorn causes Fizo, Blorn causes P...",no,backdoor,639,"{'graph_id': 'graph9', 'model_id': 198, 'query..."


In [56]:
questions = dataset_backdoor["question"].values.tolist()
contexts = dataset_backdoor["given_info"].values.tolist()
true_answers = dataset_backdoor["answer"].values.tolist()

In [57]:
prompts = get_prompt(intstruction,questions,contexts)

In [58]:
prompts[0]

'Instruction: Answer the question strictly using the causal facts provided. Do not introduce any new information. For counterfactuals, simulate the scenario along the causal chain step by step. Only output the final answer as Yes or No. Do not write anything else.\nContext:We know that Blaf causes Ziklo, Ziklo or Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo. Blaf~ Bern(0.2). We observed Trune\nQuestion:Would Lumbo occur if not Ziklo instead of Ziklo?\nAnswer:'

In [59]:
predictions = training(prompts)

Iteration 0

Response:  Yes
Iteration 1

Response:  No
Iteration 2

Response: No
Iteration 3

Response: No
Iteration 4

Response:  No
Iteration 5

Response: No
Iteration 6

Response:  No
Iteration 7

Response: Yes
Iteration 8

Response: No
Iteration 9

Response: Yes
Iteration 10

Response: No
Iteration 11

Response:  No
Iteration 12

Response: No
Iteration 13

Response: No
Iteration 14

Response:  Yes
Iteration 15

Response: Yes
Iteration 16

Response:  No
Iteration 17

Response: No
Iteration 18

Response:  No
Iteration 19

Response:  No
Iteration 20

Response:  No
Iteration 21

Response: No
Iteration 22

Response: No
Iteration 23

Response:  No
Iteration 24

Response:  No
Iteration 25

Response: No
Iteration 26

Response:  No
Iteration 27

Response:  No
Iteration 28

Response: Yes
Iteration 29

Response:  No
Iteration 30

Response: Yes
Iteration 31

Response: Yes
Iteration 32

Response: Yes
Iteration 33

Response: No
Iteration 34

Response: No
Iteration 35

Response: Yes
Iteration 36


In [60]:
predictions_new = [elem.lower() for elem in predictions]

In [61]:
print(classification_report(true_answers,predictions_new,zero_division=0))

              precision    recall  f1-score   support

          no       0.60      0.80      0.69       105
         yes       0.65      0.41      0.50        95

    accuracy                           0.61       200
   macro avg       0.62      0.61      0.59       200
weighted avg       0.62      0.61      0.60       200

